In [20]:
from ipywidgets import Textarea
import io

if 'open' in globals():
    del open
if 'input' in globals():
    del input

original_open = open

class custom_open():
    def __init__(self):
        self.text = ''

    def __call__(self, file, *args, **kwargs):
        if file == 0:
            return io.StringIO(self.text)
        return original_open(file, *args, **kwargs)

    def updater(self, change):
        self.text = change["new"]

class custom_input():
    def __init__(self):
        self.__sio = io.StringIO('')
        self.shell = get_ipython()
        if self.shell.events.callbacks['pre_run_cell'] != []:
            self.shell.events.callbacks['pre_run_cell'] = []
        self.shell.events.register('pre_run_cell', self.pre_run_cell)

    def __call__(self):
        return self.__sio.readline().strip()

    def pre_run_cell(self, info):
        text = self.shell.user_ns.get('text_area', None).value
        self.__sio = io.StringIO(text)

open = custom_open()
input = custom_input()

text_area = Textarea()
text_area.observe(open.updater, names='value')
display(text_area)

Textarea(value='')

In [67]:
def pow_mod(n,k,m):
    if k == 0:
        return 1
    elif k%2 == 1:
        return pow_mod(n,k-1,m)*n%m
    else:
        return pow_mod(n,k//2,m)**2%m
    

def gcd(a,b):
    a = abs(a)
    b = abs(b)
    if b == 0:
        return a
    else:
        return gcd(b,a%b)
    
def is_prime(n):
    """Miller--Rabin 素数判定法"""
    if n <= 1: 
        return False
    elif n == 2 or n == 3:
        return True
    elif n%2 == 0:
        return False
    else:
        tests = [2, 325, 9375, 28178, 450775, 9780504, 1795265022]
        s = 0
        d = n-1
        while d%2 == 0:
            s += 1
            d >>= 1
        for a in tests:
            if a%n == 0:
                return True
            x = pow_mod(a,d,n)
            if x != 1:
                for t in range(s+1):
                    if x == n-1:
                        break
                    x = (x*x)%n
                if t == s:
                    return False
        return True
    
def pollard(n):
    """
    Polardのrho素因数分解法
    nの素因数を返す
    """
    if n%2 == 0:
        return 2
    if is_prime(n):
        return n
    def f(x):
        return (x**2 + 1)%n
    step = 0
    while True:
        step += 1
        x = step
        y = f(x)
        while True:
            p = gcd(y-x+n,n)
            if p == 0 or p == n:
                break
            if p != 1:
                return p
            x = f(x)
            y = f(f(y))

def prime_factorise(n):
    if n == 1:
        return []
    p = pollard(n)
    if p == n:
        return [p]
    left = prime_factorise(p)
    right = prime_factorise(n//p)
    left += right
    left.sort()
    return left
    
from collections import Counter
K = int(input())
P = prime_factorise(K)
C = Counter(P)
ans = []
for p in C:
    cnt = 0
    for i in range(1,C[p]+1):
        x = i*p
        while x%p == 0:
            x //= p
            cnt += 1
        if C[p] <= cnt:
            ans.append(i*p)
            break
print(max(ans))

5
